Perfect! Let’s design a **full workflow where both the parent LLM (`main.py`) and MCP server LLM** perform reasoning. The idea:

1. **Parent LLM**:

   * Reads the user query.
   * Determines all relevant locations, ZIP codes, countries, districts.
   * Determines forecast types (today, tomorrow, 3-day, 7-day).
   * Passes a structured request to the MCP server.

2. **MCP Server LLM**:

   * Receives a location/forecast request.
   * Optionally adjusts/validates the request (e.g., normalizes city names, fetches exact ZIP codes if needed).
   * Calls real weather API (like `wttr.in`) and returns a structured result.

This ensures **double reasoning**, making the system smarter and more flexible.

---

### **`weather_mcp_server.py`**

````python
# weather_mcp_server.py
from fastmcp.server import FastMCP
from langchain_openai import ChatOpenAI
import requests
import os
import json

mcp = FastMCP("weather-mcp")

# MCP internal LLM
mcp_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5,
                     api_key=os.getenv("OPENAI_API_KEY"))

@mcp.tool()
def get_weather(location: str, forecast_type: str = "today"):
    """
    MCP server tool:
    1. Use internal LLM to validate/adjust location and forecast_type.
    2. Call weather API (wttr.in).
    """
    try:
        # LLM reasoning inside MCP
        reasoning_prompt = f"""
        You are a weather assistant inside MCP server.
        Adjust the following location and forecast request to a standard format.
        Input: location='{location}', forecast_type='{forecast_type}'
        Output strictly as JSON: {{"location": "validated_location", "forecast_type": "validated_forecast_type"}}
        Valid forecast types: today, tomorrow, 3-day, 7-day
        """
        validated = mcp_llm.invoke(reasoning_prompt).strip()
        if validated.startswith("```"):
            validated = validated.strip("`").split("json")[-1].strip()
        try:
            validated_json = json.loads(validated)
            location = validated_json.get("location", location)
            forecast_type = validated_json.get("forecast_type", forecast_type)
        except Exception:
            pass

        # Call wttr.in
        suffix_map = {"today": "?1", "tomorrow": "?2", "3-day": "?3", "7-day": "?7"}
        suffix = suffix_map.get(forecast_type.lower(), "?1")
        url = f"https://wttr.in/{location}{suffix}?format=3"
        weather_resp = requests.get(url, timeout=10).text

        return {
            "location": location,
            "forecast_type": forecast_type,
            "content": weather_resp
        }

    except Exception as e:
        return {"error": str(e)}

if __name__ == "__main__":
    mcp.run()
````

---

### **`weather_client.py`**

```python
# weather_client.py
import os
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

class WeatherMCPClient:
    """
    MCP client for weather server
    """
    def __init__(self):
        self._session = None

    async def init_session(self):
        if self._session is None:
            current_dir = os.path.dirname(os.path.abspath(__file__))
            server_path = os.path.join(current_dir, "weather_mcp_server.py")
            server_params = StdioServerParameters(command="python", args=[server_path])
            self._streams = await stdio_client(server_params).__aenter__()
            self._session = await ClientSession(*self._streams).__aenter__()
            await self._session.initialize()
        return self._session

    async def get_weather(self, location: str, forecast_type: str = "today"):
        session = await self.init_session()
        result = await session.call_tool("get_weather", {"location": location, "forecast_type": forecast_type})
        return result
```

---

### **`main.py`**

````python
# main.py
import asyncio
from weather_client import WeatherMCPClient
from langgraph.graph import StateGraph, END
from typing_extensions import Annotated
from langchain_openai import ChatOpenAI
import os
import json

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5,
                 api_key=os.getenv("OPENAI_API_KEY"))

class AgentState(dict):
    query: str
    targets: dict           # {"weather": [{"location": "Paris", "forecast_type": "today"}]}
    results: Annotated[dict, "aggregate"]
    final: str

# --- Node 1: Parent LLM classifies user query ---
async def classify_query(state: AgentState) -> AgentState:
    prompt = f"""
    You are a world-class weather assistant.
    Extract all relevant locations, zip codes, countries, districts and forecast types (today, tomorrow, 3-day, 7-day)
    from this user query: "{state['query']}"
    Output strictly as JSON like:
    {{
        "weather": [
            {{"location": "Paris", "forecast_type": "today"}}
        ]
    }}
    """
    content = llm.invoke(prompt).strip()
    if content.startswith("```"):
        content = content.strip("`").split("json")[-1].strip()
    try:
        parsed = json.loads(content)
    except Exception:
        parsed = {"weather": [{"location": state["query"], "forecast_type": "today"}]}
    state["targets"] = parsed
    state["results"] = {}
    return state

# --- Node 2: Call weather MCP ---
def call_weather_node():
    async def node(state: AgentState) -> AgentState:
        client = WeatherMCPClient()
        weather_targets = state.get("targets", {}).get("weather", [])
        results = {}
        for item in weather_targets:
            loc = item.get("location")
            forecast = item.get("forecast_type", "today")
            res = await client.get_weather(loc, forecast)
            results[f"{loc} ({forecast})"] = res.get("content")
        return {"results": {"weather": results}}
    return node

# --- Node 3: merge results ---
async def merge_results(state: AgentState) -> AgentState:
    state["final"] = state.get("results", {})
    return state

# --- Build workflow ---
workflow = StateGraph(AgentState)
workflow.add_node("classify", classify_query)
workflow.set_entry_point("classify")

workflow.add_node("weather", call_weather_node())
workflow.add_edge("classify", "weather")

workflow.add_node("merge", merge_results)
workflow.add_edge("weather", "merge")
workflow.add_edge("merge", END)

app = workflow.compile()

# --- Run ---
async def main():
    query = input("You: ")
    result = await app.ainvoke({"query": query})
    print("🌍 Weather Result:\n", result["final"])

if __name__ == "__main__":
    asyncio.run(main())
````

---

### ✅ Features:

1. **Parent LLM** in `main.py` extracts all relevant weather queries from user text.
2. **MCP LLM** in `weather_mcp_server.py` validates/normalizes each query.
3. **Supports multi-location, multi-forecast** queries.
4. **Single MCP server + single client**.
5. Returns **aggregated results in LangGraph workflow**.

---

If you want, I can **also add ZIP code resolution automatically** using the MCP LLM—so the user can type `"760001"` or `"New York"` and the MCP will normalize it to a valid location for the weather API.

Do you want me to add that next?


Here’s a list of example queries you can try with this **world-weather LangGraph + MCP setup**. The parent LLM in `main.py` will parse them intelligently and call the MCP server for each location/forecast:

---

### **Single location queries**

1. `"What’s the weather in Paris today?"`
   → Extracts: `{"weather": [{"location": "Paris", "forecast_type": "today"}]}`

2. `"Show me tomorrow’s forecast for New York"`
   → Extracts: `{"weather": [{"location": "New York", "forecast_type": "tomorrow"}]}`

3. `"Weather in 760001"` (ZIP code in India)
   → Extracts: `{"weather": [{"location": "760001", "forecast_type": "today"}]}`

---

### **Multiple locations**

4. `"Weather in London and Tokyo today"`
   → Extracts: `{"weather": [{"location": "London", "forecast_type": "today"}, {"location": "Tokyo", "forecast_type": "today"}]}`

5. `"Check tomorrow’s forecast for Paris, New Delhi, and 90210"`
   → Extracts: `{"weather": [{"location": "Paris", "forecast_type": "tomorrow"}, {"location": "New Delhi", "forecast_type": "tomorrow"}, {"location": "90210", "forecast_type": "tomorrow"}]}`

---

### **Different forecast types**

6. `"Give me 3-day forecast for Berlin and 7-day forecast for Mumbai"`
   → Extracts: `{"weather": [{"location": "Berlin", "forecast_type": "3-day"}, {"location": "Mumbai", "forecast_type": "7-day"}]}`

7. `"How is the weather in San Francisco this week?"`
   → Parent LLM can interpret `"this week"` → `"7-day"` forecast.

---

### **Mixed location types**

8. `"Tell me weather in Paris, New York, ZIP 10001, and London"`
   → Handles cities + ZIP codes together.

9. `"Current weather in Tokyo district Shibuya and Paris 18th arrondissement"`
   → Parent LLM extracts districts or sub-locations intelligently.

---

### **Fun / complex queries**

10. `"I’m planning to travel from Sydney to Auckland tomorrow. What’s the weather like?"`
    → Parent LLM detects two locations and sets forecast_type `"tomorrow"`.

11. `"Show me today’s weather for multiple cities: Paris, Berlin, Tokyo, and Delhi"`

---

The **key idea**:

* **Parent LLM** analyzes the text and creates structured targets: location + forecast type.
* **MCP server LLM** validates/normalizes each location/forecast before calling the weather API.
* The workflow supports **aggregating multiple results**.

---

If you want, I can make a **cheat sheet of 50+ realistic weather queries** showing exactly what the parent LLM would extract as JSON targets.

Do you want me to do that?
